In [ ]:
import numpy as np
import glob
import re
import shutil
import random
import pandas as pd
import tensorflow as tf # python version should be  >3.8 and <3.12
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA 
from sklearn.decomposition import KernelPCA
import os
from pathlib import Path # For path handling

In [ ]:
# ── Path Configuration ────────────────────────────────────────────────────────
# Centralise the base path so every cell below stays consistent.
# Path.cwd() returns the folder where Jupyter was launched.
# .parent.parent navigates two levels up to reach the repository root.
ROOT = Path.cwd().parent.parent ## Where your repositpry is located

BASE = str(ROOT / "data" / "raw")
DB   = BASE + "/DataBase"

print(f"Project root : {ROOT}")
print(f"Data path    : {BASE}") # folder containing the compressed data


In [ ]:
# Unzip the raw dataset archive into the raw/ folder.
# The -d flag sets the destination directory so the extracted files land
# directly under BASE instead of the current working directory.
!unzip {BASE}/dataset_non_augmented.zip -d {BASE}


In [ ]:
# Collect paths of every file nested two levels deep inside DataBase/.
# Expected structure: DataBase/<class>/<sample>.csv
#   e.g. DataBase/SETA/healthy_open0.csv

files = glob.glob(DB + "/*/*")
print(f"Total original samples found: {len(files)}")  # expected: 48


'''
glob.glob() is a function that searches for files in the system using patterns with wildcards (*, **, ?).
Returns an structure like that:
[
  "data/raw/DataBase/SETA/sample1.csv",
  "data/raw/DataBase/SETA/sample2.csv",
  "data/raw/DataBase/AD/sample1.csv",
  ...
]
'''

## Step 1 — Data Cleaning

Some CSV columns contain **string / object** values instead of numbers. These
are caused by sensor read errors (e.g. two numbers merged into `22.56900.1`).

**Strategy:**
1. Detect columns whose dtype is `object` (i.e. not numeric).
2. Coerce invalid entries to `NaN` via `pd.to_numeric(..., errors='coerce')`.
3. Fill `NaN` with the **previous** valid value (`ffill`), then with the
   **next** valid value (`bfill`) for any remaining gaps at the start.
4. Truncate each file to exactly **1024 rows** for a uniform sample length.


In [ ]:
def clean(path: str) -> None:
    # Read a CSV, fix non-numeric columns, standardise length, and overwrite.
    df = pd.read_csv(path)

    for column in df.columns:
        # dtype 'object' signals strings are present — column needs repair
        if df[column].dtype == 'object':
            print(f"  Dirty column → file: {path}  |  column: {column}")

            # Convert to numeric; anything that cannot be parsed becomes NaN
            df[column] = pd.to_numeric(df[column], errors='coerce')

            # Forward-fill: replace NaN with the value from the previous row
            df[column] = df[column].ffill()

            # Backward-fill: handles NaN that ffill could not reach (e.g. row 0)
            df[column] = df[column].bfill()
            
            df[column] = df[column].fillna(0) 

    # Standardise every sample to exactly 1024 timesteps
    df = df.iloc[:1024, :]

    # Overwrite the original file with the cleaned version
    df.to_csv(path, index=False)


In [ ]:
# Create a full backup of the original dataset before any modification.
# This lets you restore the raw files without re-extracting the zip.
backup_path = DB + "_original"
if not __import__('os').path.exists(backup_path):
    shutil.copytree(DB, backup_path)
    print(f"Backup created at: {backup_path}")
else:
    print("Backup already exists — skipping.")


In [ ]:
import glob

# Defining the folders to clean
folders_to_clean = [
    DB + "/SETA", 
    DB + "/SETB", 
    DB + "/SETC", 
    DB + "/SETD"
]

print("Initiating data cleaning...")

for folder in folders_to_clean:
    # Pega todos os arquivos CSV dentro de cada pasta
    for path in glob.glob(folder + "/*.csv"):
        # Aplica a sua função que arruma as colunas, corta pra 1024 e sobrescreve
        clean(path)

print("✅ Limpeza concluída em todas as pastas! Agora é seguro rodar o augmentation.")

In [ ]:
# ── Sanity Check: Verify No Dirty Columns Remain After Cleaning ──────────────
# Re-reads every original file and looks for columns that are still 'object'
# dtype (i.e. contain strings instead of numbers).
# If any are found, they are collected in the 'dirty' list for inspection.
dirty = []

for file in files:
    df = pd.read_csv(file)
    for col in df.columns:
        # A column with dtype 'object' was not successfully converted to numeric
        if df[col].dtype == 'object':
            # Store the file path, column name, and up to 3 sample bad values
            # so we know exactly where and what the problem is
            dirty.append((file, col, df[col].unique()[:3]))

# Report the results
if dirty:
    # At least one dirty column was found — print each occurrence for debugging
    print("⚠️  Still dirty:")
    for f, c, v in dirty:
        print(f"  file: {f}  |  column: {c}  |  bad values: {v}")
else:
    # All columns in all files are now numeric — safe to proceed
    print("✅  All files are clean.")

### Bug Report: `TypeError: unsupported operand type(s) for -: 'float' and 'str'`

#### Problem
The dirty check passed because it only verified the `files` list — the 48 original samples.
However, `add_shift_per_sample()` used `glob.glob(folder + "/*")` which picks up **everything**
inside the folder, including corrupted augmented files generated by previous failed runs.
Those files were being fed back into the shift functions, and when `df.shift()` was called
on them, pandas silently converted some `float64` columns to `object` dtype — even when
the input was already numeric. This caused `df.std()` to crash because it tried to subtract
a float from a string

#### Root Cause
- One possibility could be that `df.shift()` introduces `NaN` values, and in some pandas versions this operation silently
casts columns to `object` dtype, regardless of the original dtype of the DataFrame.
- On the other hand, it could bem caused by the difference between how you searched for the files (but I don't thisnk that was the problem):
    - Without sanity check: glob.glob(DB + "/* /*") (this is very restrictive, searches exactly two folders inside)
    - Without Augmentation: glob.glob(folder + "/*") (gets anything inside the SETA folder)
    - If inside the SETA folder there is a hidden system file (such as .DS_Store on Mac), a README.txt text file, or any other junk that is not a valid CSV, Augmentation will try to read that file

#### Fix
Force all columns back to numeric immediately after `fillna(0)` and before `df.std()` using:
```python
df = df.apply(pd.to_numeric, errors='coerce').fillna(0)
```

This guarantees that no string values survive into the `std()` calculation, making the
shift functions robust against any dtype corruption introduced by pandas internals.

# Step 2 — Data Augmentation Techniques : 
  1.) **Shifting and Noising :** Shifting in time series data refers to a
transformation of the data where the entire series is moved forward or backward in time by a certain number of periods. This can be useful in analyzing time series data, as it can help to identify patterns or trends that may not be immediately apparent when looking at the original data. Noising adds random noise to the shifted data so that the learning model will learn to ignore noisy information and filter out the relevant info from the data.
  - **Why it works:** In real-world scenarios, an event might happen slightly earlier or later than expected. Shifting teaches the model that the pattern of the signal is more important than its exact timestamp.

 2.) **Rolling Mean :** Rolling mean is a technique used to smooth out the data by taking the average of a fixed window of data points. This technique can be useful for reducing noise in the data and identifying trends that are relevant.
  - **The Goal:** It acts as a regularizer. By training on "dirty" data, the model learns to focus on the underlying signal rather than memorizing the specific noise patterns of the training set.

<br>

> Combined, these techniques prevent overfitting. The model stops "memorizing" exact coordinates and starts "understanding" the shape of the data.

## Shifting

In [ ]:
# ── Shift Augmentation ────────────────────────────────────────────────────────

def positive_shift(path: str, replace: bool = False) -> pd.DataFrame:
    """Return a copy of the sample shifted forward in time by 3–15 steps."""
    df = pd.read_csv(path)

    # Shift all columns forward; rows at the start become NaN
    n_steps = random.randint(3, 15)
    df = df.shift(n_steps)

    # Optionally fill NaN with column means; always fall back to 0
    if replace:
        df = df.fillna(df.mean())
        df = df.fillna(0)
    
    # Force all columns to numeric before std() — shift() can silently
    # change column dtypes to object in some pandas versions
    df = df.apply(pd.to_numeric, errors='coerce').fillna(0)

    # Scale noise by each channel's standard deviation so it stays realistic
    deviation = df.std().tolist() #  the moment in what the float error occurs
    noise = np.random.normal([0] * 19, deviation, 19)

    # Add 10 % of the noise amplitude to the signal
    df += noise / 10
    return df


def negative_shift(path: str, replace: bool = False) -> pd.DataFrame:
    """Return a copy of the sample shifted backward in time by 3–15 steps."""
    df = pd.read_csv(path)

    # Shift all columns backward; rows at the end become NaN
    n_steps = random.randint(-15, -3)
    df = df.shift(n_steps)

    if replace:
        df = df.fillna(df.mean())
    df = df.fillna(0)
    
    # Same fix as positive_shift
    df = df.apply(pd.to_numeric, errors='coerce').fillna(0)

    deviation = df.std().tolist()
    noise = np.random.normal([0] * 19, deviation, 19)
    df += noise / 10
    return df


def apply_shift(path: str, label: str, file_no: int, target_dir: str) -> None:
    """Randomly choose positive or negative shift and write the result."""
    # 50 / 50 chance between forward and backward shift
    use_positive = random.randint(0, 1)

    out_path = f"{target_dir}/{label}{file_no}.csv"

    if use_positive:
        df = positive_shift(path)
        direction = "Positive"
    else:
        df = negative_shift(path)
        direction = "Negative"

    print(f"{path}  →  {out_path}  [{direction} Shift]")
    df.to_csv(out_path, index=False)


def add_shift_per_sample(folder: str, label: str, start_idx: int = 25) -> None:
    """Apply one shift augmentation per original sample in the folder."""
    # Augmented files are numbered from start_idx onward (originals are 0-24)
    file_no = start_idx
    for path in glob.glob(folder + "/*"):
        apply_shift(path, label, file_no, folder)
        file_no += 1


In [ ]:
# Apply shift augmentation to all four EEG condition classes
add_shift_per_sample(DB + "/SETA", "healthy_open")
add_shift_per_sample(DB + "/SETB", "healthy_closed")
add_shift_per_sample(DB + "/SETC", "alzeimer_open")
add_shift_per_sample(DB + "/SETD", "alzeimer_closed")


## Rolling Mean

In [ ]:
# ── Rolling-Mean Augmentation ─────────────────────────────────────────────────
# FIX: the original code computed a random window but never used it (always
# passed window=5 to rolling()). Now the random window is actually applied.

def mean_roll(path: str, label: str, file_no: int, target_dir: str) -> None:
    """Smooth the sample with a random rolling window and save as a new file."""
    df = pd.read_csv(path)
    
    
        
    # The `df.rolling().mean()` function also corrupted the dtype, just like `shift()`. Applied the same fix:
    # Force all columns to numeric RIGHT AFTER reading — before any operation — rolling().mean() can silently cast
    # columns to object dtype in some pandas versions, just like shift()
    df = df.apply(pd.to_numeric, errors='coerce').fillna(0)

    # Random window size between 3 and 8 timesteps
    window = random.randint(3, 8)

    # center=True: the window is centred on each point (reduces border artefacts)
    df = df.rolling(window=window, center=True).mean()

    # Forward/backward fill any NaN left at the series edges, then zero-fill
    df = df.ffill().bfill().fillna(0)


    out_path = f"{target_dir}/{label}{file_no}.csv"
    print(f"{path}  →  {out_path}  [window={window}]")
    df.to_csv(out_path, index=False)


def add_rolling_per_sample(folder: str, label: str, start_idx: int = 49) -> None:
    """Apply one rolling-mean augmentation per original sample in the folder."""
    # Rolling samples start at index 49 (shift samples occupy 25-48)
    file_no = start_idx
    for path in glob.glob(folder + "/*"):
        mean_roll(path, label, file_no, folder)
        file_no += 1


In [ ]:
# Apply rolling-mean augmentation to all four EEG condition classes
add_rolling_per_sample(DB + "/SETA", "healthy_open")
add_rolling_per_sample(DB + "/SETB", "healthy_closed")
add_rolling_per_sample(DB + "/SETC", "alzeimer_open")
add_rolling_per_sample(DB + "/SETD", "alzeimer_closed")


# Step 3 — Splitting data to train and test

**Strategy:** keep all *augmented* files for training and use the original 48
samples for testing — but also move 1/3 of the originals into training to
ensure the model sees some clean, unaugmented examples during training.

Specifically, for every 12 consecutive original files, the first 4 are added
to the training set; the remaining 8 stay in the test set.

In [ ]:
def return_splits():
    """Split augmented + original files into training and test sets.

    Returns
    -------
    train_files : list[str]  All augmented files + 1/3 of the originals.
    test_files  : list[str]  Remaining 2/3 of the original samples.
    """
    # All files currently on disk (originals + augmented)
    augmented_files = glob.glob(DB + "/*/*")

    # Augmented-only files = full set minus the original 48
    train_files = list(set(augmented_files) - set(files))

    # Add the first 4 out of every 12 original samples to training
    test_files = files.copy()
    for i in range(0, len(test_files), 12):
        train_files += test_files[i:i + 4]

    # Test set = originals not promoted to training
    test_files = list(set(test_files) - set(train_files))
    return train_files, test_files


train, test = return_splits()
print(f"Training samples : {len(train)}")
print(f"Test samples     : {len(test)}")


# Step 4 — Time-Series Compression with KernelPCA


### The Problem
Each EEG sample is a matrix of **(1024 timesteps × 19 channels)**.
Feeding 19,456 raw numbers per sample directly into a ML model would be:
- **Slow** — too many numbers to process
- **Inaccurate** — most ML models struggle when there are more features than samples
  (known as the "curse of dimensionality")

### The Intuition
Think of it like summarising a 2-hour movie into a 5-sentence synopsis.
You lose some detail, but the essential story is preserved.
KernelPCA finds the **most important patterns** across the 19 EEG channels
and discards the rest.

### What Actually Happens
1. The matrix is **transposed** → now we have 19 channels, each described by 1024 values
2. **KernelPCA** looks at those 19 channels and finds 5 directions that best
   explain the differences between them (the "principal components")
3. The mean of those 5 components gives us a **single vector of 5 numbers**
   that summarises the entire EEG recording

### Why RBF Kernel?
A regular PCA only captures linear relationships. The **RBF (Radial Basis Function)**
kernel allows KernelPCA to capture **non-linear patterns** — which are common in
brain signals since neurons don't interact in simple straight-line relationships.

### The Result
| Before | After |
|---|---|
| 1024 × 19 = 19,456 numbers | 5 numbers |
| Hard for ML models to process | Fast and efficient |
| Contains noise and redundancy | Contains the dominant signal |

In [ ]:
def decompose(wave_data: np.ndarray) -> list:
    """Compress a (1024, 19) EEG matrix to a feature vector.

    NOTE: mean(axis=1) returns 19 values (one per channel), not 5.
    Is this a bug? The original model
    was trained this way — changing axis to 0 would alter results.

    Returns a list of 19 values, which combined with 2 label columns
    produces the final dataset shape: (n_samples, 21)
    """
    # Transpose: shape becomes (19, 1024) — channels as samples
    time_series_data = wave_data.T

    # RBF kernel captures non-linear relationships between channels
    kpca = KernelPCA(n_components=5, kernel='rbf', gamma=0.1)
    
    # fit_transform output: (19, 5) — 19 channels × 5 components
    # mean(axis=1) → (19,)  one value per channel  ← current behaviour (bug?)
    # mean(axis=0) → (5,)   one value per component ← correct behaviour
    compressed_data = kpca.fit_transform(time_series_data).mean(axis=1)
    # "The correct" implementation would be:
    # compressed_data = kpca.fit_transform(time_series_data).mean(axis=0
    
    return compressed_data.tolist() # returns 19 values per sample


In [ ]:

def handle_nan(df_f: str) -> pd.DataFrame:
    """Load a CSV and iteratively fill any remaining NaN values."""
    df = pd.read_csv(df_f)
    
    df = df.apply(pd.to_numeric, errors='coerce').fillna(0) # same problem of corrupted dtype occurs here as well, just like in shift() and rolling().mean()

    if df.isnull().sum().sum() != 0:
        count = 0
        print(df_f, end=" ")

        # Repeat until no NaN remains (handles edge cases missed by a single pass)
        while df.isnull().sum().sum() != 0:
            count += 1
            df = df.ffill().bfill()  # Updated from deprecated fillna(method=...)

        print(f"  → {count} NaN pass(es) applied")
    return df



In [ ]:
def create_ds(all_files: list) -> pd.DataFrame:
    """Build a feature-matrix DataFrame from a list of EEG CSV files.

    Each row = one sample.  The last two columns encode:
      - eye condition  : 'closed' or 'open'
      - cognitive state: 'healthy' or 'alzeimer'
    """
    dataset = []

    for file in all_files:
        # Clean NaN, then standardise each channel to zero mean / unit variance
        df = handle_nan(file)
        
        scaler = StandardScaler()
        df_scaled = scaler.fit_transform(df)

        # Compress to a 5-element feature vector
        sample = decompose(df_scaled)

        # Derive labels from the file name
        sample.append("closed" if "closed" in file else "open")
        sample.append("alzeimer" if "alzeimer" in file else "healthy")

        dataset.append(sample)

    return pd.DataFrame(dataset)

In [ ]:
# Build the training feature matrix and save to CSV
print("Building training dataset...")
train_df = create_ds(train)
train_df.to_csv("/home/georis/temp/alzheimers-detection-methodologies-organized/data/processed/train.csv", index=False)
print(f"train_augmented.csv saved — shape: {train_df.shape}")


In [ ]:

# Build the test feature matrix and save to CSV
print("Building test dataset...")
test_df = create_ds(test)
test_df.to_csv("/home/georis/temp/alzheimers-detection-methodologies-organized/data/processed/test.csv", index=False)
print(f"test_augmented.csv saved  — shape: {test_df.shape}")